In [ ]:
!pip install TTS
!apt-get -qq install -y ffmpeg
!apt-get -qq install -y tesseract-ocr ffmpeg
!pip install easyocr pytesseract gtts opencv-python-headless pydub language_tool_python
!pip install git+https://github.com/csebuetnlp/normalizer
!pip install transformers torch

!apt-get -qq install -y ffmpeg
!pip install easyocr gTTS ipywidgets pydub torch sentencepiece
!pip install huggingface_hub
!apt-get install tesseract-ocr-ben
# And install normalizer if needed (not used here, but if you want text normalization):
# !pip install git+https://github.com/csebuetnlp/normalizer

In [ ]:
###############################################################################
# Imports & Setup
###############################################################################
import os
import re
import io
import shutil
import base64
import math
import statistics
import numpy as np

from IPython.display import display, HTML
from PIL import Image

import cv2  # For optional image processing if desired

import pytesseract  # Tesseract-based OCR

# Audio processing
from pydub import AudioSegment
import soundfile as sf  # For saving synthesized wav
from scipy.io.wavfile import write as scipy_wav_write

# TTS Imports – using the TTS library's Synthesizer
from TTS.utils.synthesizer import Synthesizer
import torch # Import torch for device management

# Global sampling rate for TTS output
DEFAULT_SAMPLING_RATE = 22050

###############################################################################
# Custom TTS Engine Wrapper
###############################################################################
class TextToSpeechEngine:
    def __init__(self, models):
        self.models = models

    def infer_from_text(self, input_text, lang, speaker_name):
        synthesizer = self.models.get(lang)
        if synthesizer is None:
            raise ValueError(f"No synthesizer found for language: {lang}")
        # Call the synthesizer's tts method.
        # Ensure input is string and let TTS library handle device placement
        return synthesizer.tts(input_text, speaker_name=speaker_name)

###############################################################################
# OCR: Extract Text from Entire Image using pytesseract
###############################################################################
def extract_text(image_path):
    """
    Opens the image and uses pytesseract to extract Bengali text.
    """
    image = Image.open(image_path)
    text = pytesseract.image_to_string(image, lang='ben')
    return text

###############################################################################
# Sentence & Paragraph Splitting Helpers
###############################################################################
def split_into_sentences_bn(text):
    split_re = re.compile(r'([।\?!])')
    parts = split_re.split(text)
    if not parts:
        return [text.strip()]
    combined = []
    temp = ""
    for i in range(len(parts)):
        if i % 2 == 0:
            temp += parts[i]
        else:
            temp += parts[i]
            combined.append(temp.strip())
            temp = ""
    if temp.strip():
        combined.append(temp.strip())
    final_sentences = []
    skip = False
    for idx, s in enumerate(combined):
        if skip:
            skip = False
            continue
        if len(s) < 5 and idx < len(combined) - 1:
            merged = s + " " + combined[idx+1]
            final_sentences.append(merged.strip())
            skip = True
        else:
            final_sentences.append(s.strip())
    final_sentences = [s for s in final_sentences if s]
    return final_sentences

def merge_too_short_sentences(sents, min_len=3):
    merged = []
    skip = False
    for i, s in enumerate(sents):
        if skip:
            skip = False
            continue
        if len(s) < min_len and i < len(sents) - 1:
            merged.append((s + " " + sents[i+1]).strip())
            skip = True
        else:
            merged.append(s)
    return merged

def build_paragraphs_and_sentences_from_text(full_text):
    lines = [line.strip() for line in full_text.splitlines() if line.strip()]
    paragraphs = []
    current_paragraph = []
    for line in lines:
        current_paragraph.append(line)
        if line.endswith(('।', '?', '!', '.')):
            paragraphs.append(" ".join(current_paragraph))
            current_paragraph = []
    if current_paragraph:
        paragraphs.append(" ".join(current_paragraph))

    paragraph_objs = []
    for idx, para in enumerate(paragraphs):
        sents = split_into_sentences_bn(para)
        sents = merge_too_short_sentences(sents, min_len=3)
        sentence_objs = [{"text": s.strip()} for s in sents if s.strip()]
        paragraph_objs.append({
            "paragraph_idx": idx,
            "sentences": sentence_objs
        })
    return paragraph_objs

###############################################################################
# Bengali TTS Engine Initialization using TTS Library
###############################################################################
def init_tts_engine():
    # Create required directory structure
    os.makedirs('/content/models/v1/bn/fastpitch/', exist_ok=True)

    # Copy speaker file to expected location
    shutil.copy('/content/speakers.pth', '/content/models/v1/bn/fastpitch/speakers.pth')

    lang = "bn"
    device = "cuda" if torch.cuda.is_available() else "cpu" # Determine device

    bn_model = Synthesizer(
        tts_checkpoint="/content/drive/MyDrive/models/v1/bn/fastpitch/best_model_fp.pth",
        tts_config_path="/content/config_fp.json",
        tts_speakers_file="/content/models/v1/bn/fastpitch/speakers.pth",
        tts_languages_file=None,
        vocoder_checkpoint="/content/drive/MyDrive/models/v1/bn/hifigan/best_model.pth",
        vocoder_config="/content/config.json",
        use_cuda=device == "cuda", # Use determined device
    )

    # ====== Ensure all relevant components move to determined device ======
    bn_model.tts_model.to(device)
    if hasattr(bn_model, "vocoder"):
        bn_model.vocoder.to(device)
    elif hasattr(bn_model, "vocoder_model"):
        bn_model.vocoder_model.to(device)
    else:
        print("Warning: No vocoder attribute found, skipping vocoder GPU assignment.")

    if bn_model.speaker_manager is not None and bn_model.speaker_manager.embedding_function is not None:
        bn_model.speaker_manager.embedding_function.to(device)
    # =================================================================================

    models = {lang: bn_model}
    engine = TextToSpeechEngine(models)
    return engine

###############################################################################
# Word-Level TTS Generation
###############################################################################
def generate_word_level_tts(paragraphs, tts_engine, base_folder="tts_word_segments"):
    if not os.path.exists(base_folder):
        os.makedirs(base_folder)

    sentence_meta = []
    global_sentence_index = 0

    for p_obj in paragraphs:
        p_idx = p_obj['paragraph_idx']
        for s_idx, s_obj in enumerate(p_obj['sentences']):
            txt = s_obj['text'].strip()

            # Remove problematic characters from text before TTS
            txt = txt.replace('_', '') # Remove underscore

            if not txt:
                continue

            print(f"  > Text splitted to sentences.")
            print([txt])
            print(txt)


            filename_wav = os.path.join(base_folder, f"sent_{global_sentence_index}.wav")
            try:
                raw_audio = tts_engine.infer_from_text(
                    input_text=txt,
                    lang="bn",
                    speaker_name="male"
                )
                scipy_wav_write(filename_wav, DEFAULT_SAMPLING_RATE, raw_audio)
            except Exception as e:
                print(f"Pipeline failed during TTS inference for sentence: '{txt}'. Error: {str(e)}")
                raise e # Re-raise to stop the pipeline

            audio_segment = AudioSegment.from_wav(filename_wav).set_channels(1)
            filename_mp3 = os.path.join(base_folder, f"sent_{global_sentence_index}.mp3")
            audio_segment.export(filename_mp3, format="mp3")
            sentence_duration_ms = len(audio_segment)

            words = txt.split()
            total_chars = sum(len(w) for w in words) or 1

            current_start = 0
            local_word_info = []
            for w_i, w_text in enumerate(words):
                w_len = len(w_text)
                portion = int((w_len / total_chars) * sentence_duration_ms)
                w_start = current_start
                w_end = w_start + portion
                local_word_info.append({
                    'paragraph_idx': p_idx,
                    'sentence_idx': s_idx,
                    'word_idx': w_i,
                    'word_text': w_text,
                    'rel_start_ms': w_start,
                    'rel_end_ms': w_end
                })
                current_start += portion

            trailing_pause_ms = 500 if any(c in txt for c in ['?', '!', '।']) else 300

            sentence_meta.append({
                'paragraph_idx': p_idx,
                'sentence_idx': s_idx,
                'text': txt,
                'file_path': filename_mp3,
                'pause_ms': trailing_pause_ms,
                'words_info': local_word_info
            })

            global_sentence_index += 1

    return sentence_meta

###############################################################################
# Audio Combination and Player Setup
###############################################################################
def combine_word_level_segments(sentence_meta, output_file="combined_audio.mp3"):
    combined = AudioSegment.empty().set_channels(1)
    current_start_ms = 0
    merged_data = []

    for sdata in sentence_meta:
        seg = AudioSegment.from_mp3(sdata['file_path']).set_channels(1)
        seg_len = len(seg)

        for w_info in sdata['words_info']:
            global_ws = current_start_ms + w_info['rel_start_ms']
            global_we = current_start_ms + w_info['rel_end_ms']
            merged_data.append({
                'paragraph_idx': w_info['paragraph_idx'],
                'sentence_idx': w_info['sentence_idx'],
                'word_idx': w_info['word_idx'],
                'start_time_sec': global_ws / 1000.0,
                'end_time_sec': global_we / 1000.0,
                'text': w_info['word_text']
            })

        combined += seg
        current_start_ms += seg_len

        pm = sdata['pause_ms']
        if pm > 0:
            combined += AudioSegment.silent(duration=pm)
            current_start_ms += pm

    combined.export(output_file, format="mp3")
    return output_file, merged_data

def build_keyboard_player(audio_file, merged_data, default_speed=1.0):
    with open(audio_file, "rb") as f:
        audio_bin = f.read()
    b64_audio = base64.b64encode(audio_bin).decode("utf-8")

    paragraphs = []
    sentences = []
    words = []
    starts = []
    ends = []

    for item in merged_data:
        paragraphs.append(item['paragraph_idx'])
        sentences.append(item['sentence_idx'])
        words.append(item['word_idx'])
        starts.append(f"{item['start_time_sec']:.4f}")
        ends.append(f"{item['end_time_sec']:.4f}")

    js_paragraphs = "[" + ", ".join(map(str, paragraphs)) + "]"
    js_sentences = "[" + ", ".join(map(str, sentences)) + "]"
    js_words = "[" + ", ".join(map(str, words)) + "]"
    js_starts = "[" + ", ".join(starts) + "]"
    js_ends = "[" + ", ".join(ends) + "]"

    html_code = f"""
<div>
    <audio id="tts-audio" preload="auto">
        <source src="data:audio/mp3;base64,{b64_audio}" type="audio/mp3">
        Your browser does not support the audio element.
    </audio>
    <p><strong>Keyboard Controls:</strong></p>
    <ul>
        <li>Space: Play/Pause</li>
        <li>Arrow Right: Next Word</li>
        <li>Arrow Left: Previous Word</li>
        <li>Arrow Down: Next Sentence</li>
        <li>Arrow Up: Previous Sentence</li>
        <li>PageDown: Next Paragraph</li>
        <li>PageUp: Previous Paragraph</li>
        <li>Shift + Right Arrow: Skip +5s</li>
        <li>Shift + Left Arrow: Skip -5s</li>
        <li>= : Increase speed</li>
        <li>- : Decrease speed</li>
    </ul>
</div>
<script>
    var audio = document.getElementById('tts-audio');
    audio.controls = false;
    var paragraphIdx = {js_paragraphs};
    var sentenceIdx  = {js_sentences};
    var wordIdx      = {js_words};
    var startTimes     = {js_starts};
    var endTimes       = {js_ends};
    var totalChunks  = startTimes.length;
    var currentIndex = 0;
    audio.playbackRate = {default_speed};

    function jumpToIndex(idx){{
        if(idx<0) idx=0;
        if(idx>=totalChunks) idx=totalChunks-1;
        currentIndex = idx;
        audio.currentTime = parseFloat(startTimes[currentIndex]);
    }}

    function nextWord(){{ jumpToIndex(currentIndex+1); }}
    function prevWord(){{ jumpToIndex(currentIndex-1); }}
    function nextSentence(){{
        var cs = sentenceIdx[currentIndex];
        for(var i = currentIndex+1; i<totalChunks; i++) {{
            if(sentenceIdx[i]>cs) {{
                jumpToIndex(i);
                return;
            }}
        }}
        jumpToIndex(totalChunks-1);
    }}
    function prevSentence(){{
        var cs = sentenceIdx[currentIndex];
        for(var i=currentIndex-1; i>=0; i--) {{
            if(sentenceIdx[i]<cs) {{
                jumpToIndex(i);
                return;
            }}
        }}
        jumpToIndex(0);
    }}
    function nextParagraph(){{
        var cp = paragraphIdx[currentIndex];
        for(var i=currentIndex+1; i<totalChunks; i++) {{
            if(paragraphIdx[i]>cp) {{
                jumpToIndex(i);
                return;
            }}
        }}
        jumpToIndex(totalChunks-1);
    }}
    function prevParagraph(){{
        var cp = paragraphIdx[currentIndex];
        for(var i=currentIndex-1; i>=0; i--) {{
            if(paragraphIdx[i]<cp) {{
                jumpToIndex(i);
                return;
            }}
        }}
        jumpToIndex(0);
    }}

    document.addEventListener('keydown', function(e){{
        if(e.code==='Space'){{
            e.preventDefault();
            if(audio.paused) audio.play(); else audio.pause();
        }}
        else if(e.code==='ArrowRight' && !e.shiftKey){{
            e.preventDefault();
            nextWord();
        }}
        else if(e.code==='ArrowLeft' && !e.shiftKey){{
            e.preventDefault();
            prevWord();
        }}
        else if(e.code==='ArrowDown'){{
            e.preventDefault();
            nextSentence();
        }}
        else if(e.code==='ArrowUp'){{
            e.preventDefault();
            prevSentence();
        }}
        else if(e.code==='PageDown'){{
            e.preventDefault();
            nextParagraph();
        }}
        else if(e.code==='PageUp'){{
            e.preventDefault();
            prevParagraph();
        }}
        else if(e.shiftKey && e.code==='ArrowRight'){{
            e.preventDefault();
            audio.currentTime += 5;
        }}
        else if(e.shiftKey && e.code==='ArrowLeft'){{
            e.preventDefault();
            audio.currentTime = Math.max(0, audio.currentTime - 5);
        }}
        else if(e.key==='='){{
            e.preventDefault();
            audio.playbackRate = Math.round((audio.playbackRate + 0.1)*10)/10;
        }}
        else if(e.key==='-'){{
            e.preventDefault();
            var nr = audio.playbackRate - 0.1;
            if(nr<0.1) nr=0.1;
            audio.playbackRate = Math.round(nr*10)/10;
        }}
    }});

    audio.addEventListener('timeupdate', function(){{
        if(currentIndex < totalChunks-1){{
            var et = parseFloat(endTimes[currentIndex]);
            if(audio.currentTime >= et){{
                currentIndex++;
            }}
        }}
    }});
</script>
"""

    return HTML(html_code)

###############################################################################
# Main Pipeline using Bengali TTS and pytesseract OCR
###############################################################################
def main_pipeline_tts():
    print("========== Enhanced OCR + Bengali TTS Pipeline ==========")
    try:
        # For Colab users: either use file upload or provide a local path.
        try:
            from google.colab import files
            choice = input("Type 'U' to Upload or provide a local path: ").strip()
            if choice.upper() == 'U':
                print("Please upload your image file...")
                up = files.upload()
                img_path = list(up.keys())[0] if up else None
            else:
                img_path = choice
        except ImportError:
            img_path = input("Enter image path: ").strip()

        if not img_path or not os.path.exists(img_path):
            print(f"ERROR: File '{img_path}' not found.")
            return

        print("\n1) Extracting text using pytesseract...")
        full_text = extract_text(img_path)
        if not full_text.strip():
            print("No text recognized in the image.")
            return

        print("\n2) Building paragraphs & sentences from extracted text...")
        paragraphs = build_paragraphs_and_sentences_from_text(full_text)
        print("\n=== Final Recognized Text ===")
        for p_obj in paragraphs:
            print(f"\nParagraph {p_obj['paragraph_idx']+1}:")
            for s in p_obj['sentences']:
                print("  •", s['text'])

        if input("\nProceed to TTS? [Y/N]: ").strip().upper() != 'Y':
            print("Exiting without TTS.")
            return

        print("\n3) Initializing Bengali TTS engine...")
        tts_engine = init_tts_engine()

        print("\n4) Generating word-level TTS segments using Bengali TTS...")
        sentence_meta = generate_word_level_tts(paragraphs, tts_engine, base_folder="tts_word_segments")

        print("5) Combining all segments into one MP3 (tracking word-level timings)...")
        final_mp3, merged_data = combine_word_level_segments(sentence_meta, output_file="combined_audio.mp3")

        print("6) Building interactive audio player with word-level navigation...")
        html_player = build_keyboard_player(final_mp3, merged_data, default_speed=1.0)
        display(html_player)

        print("\nDone! Use the keyboard shortcuts to navigate playback.")

    except Exception as e:
        print(f"Pipeline failed: {str(e)}")

###############################################################################
# Main Execution
###############################################################################
if __name__ == "__main__":
    main_pipeline_tts()